In [1]:
#Assignment 1 ai2584

In [10]:
import os

%cd /Users/Aditi_1/Documents/Columbia - Fall 2026/Applied Gen AI/Assignment 1

%mkdir app
%cd app

# Initialise the uv project once and create pyproject.toml and main.py
# Initialise the uv project once and create pyproject.toml and main.py
if not os.path.exists("pyproject.toml"):
    !uv init --python 3.12 .

/Users/Aditi_1/Documents/Columbia - Fall 2026/Applied Gen AI/Assignment 1
mkdir: app: File exists
/Users/Aditi_1/Documents/Columbia - Fall 2026/Applied Gen AI/Assignment 1/app


In [11]:
#  Create the application directory
!mkdir -p app
%cd app

/Users/Aditi_1/Documents/Columbia - Fall 2026/Applied Gen AI/Assignment 1/app/app


In [12]:
# Create the model and main application files
open("__init__.py", "w").close()

In [13]:
%%writefile bigram_model.py
import random
import re
from collections import Counter, defaultdict


class BigramModel:
    """Simple bigram language model built from a list of text strings."""

    def __init__(self, corpus):
        # bigram_counts[w1][w2] = number of times w2 follows w1
        self.bigram_counts = defaultdict(Counter)
        self.fit(corpus)

    @staticmethod
    def _tokenize(text):
        return re.findall(r"[\w']+", text.lower())

    def fit(self, corpus):
        for sentence in corpus:
            tokens = self._tokenize(sentence)
            for w1, w2 in zip(tokens, tokens[1:]):
                self.bigram_counts[w1][w2] += 1

    def generate_text(self, start_word, length):
        """Generate up to `length` words, starting from `start_word`."""
        word = start_word.lower()
        output = [word]
        for _ in range(max(length, 1) - 1):
            followers = self.bigram_counts.get(word)
            if not followers:
                break  # dead end: no known word follows this one
            words, counts = zip(*followers.items())
            word = random.choices(words, weights=counts, k=1)[0]
            output.append(word)
        return " ".join(output)


Overwriting bigram_model.py


In [14]:
# Return to project root, to main.py
%cd /Users/Aditi_1/Documents/Columbia - Fall 2026/Applied Gen AI/Assignment 1

/Users/Aditi_1/Documents/Columbia - Fall 2026/Applied Gen AI/Assignment 1


In [15]:
%%writefile main.py
from typing import Union
from fastapi import FastAPI
from pydantic import BaseModel
from app.bigram_model import BigramModel

app = FastAPI()

# import spacy
import spacy
nlp = spacy.load("en_core_web_md")

# Corpus for the bigram model
corpus = [
    "The Count of Monte Cristo is a novel written by Alexandre Dumas. "
    "It tells the story of Edmond Dantès, who is falsely imprisoned and later seeks revenge.",
    "this is another example sentence",
    "we are generating text based on bigram probabilities",
    "bigram models are simple but effective",
]

bigram_model = BigramModel(corpus)


class TextGenerationRequest(BaseModel):
    start_word: str
    length: int


class EmbeddingRequest(BaseModel):
    word: str


@app.get("/")
def read_root():
    return {"Hello": "World"}


@app.post("/generate")
def generate_text(request: TextGenerationRequest):
    generated_text = bigram_model.generate_text(request.start_word, request.length)
    return {"generated_text": generated_text}


@app.post("/embedding")
def calculate_embedding(request: EmbeddingRequest):
    vector = nlp(request.word).vector
    return {"word": request.word, "dimensions": len(vector), "embedding": vector.tolist()}


Overwriting main.py


In [19]:
# Install required dependencies (spaCy)
%cd /Users/Aditi_1/Documents/Columbia - Fall 2026/Applied Gen AI/Assignment 1
!uv add "fastapi[standard]" spacy
!uv add "en_core_web_md @ https://github.com/explosion/spacy-models/releases/download/en_core_web_md-3.8.0/en_core_web_md-3.8.0-py3-none-any.whl"

/Users/Aditi_1/Documents/Columbia - Fall 2026/Applied Gen AI/Assignment 1
error: No `pyproject.toml` found in current directory or any parent directory
error: No `pyproject.toml` found in current directory or any parent directory


In [ ]:
# 5. Sync dependencies
!uv sync

Resolved 69 packages in 291ms
Checked 67 packages in 18ms


In [ ]:
# 6. Run the application
# Run in a terminal
# uv run fastapi dev main.py

In [ ]:
from app.bigram_model import BigramModel

test_corpus = [
    "this is another example sentence",
    "we are generating text based on bigram probabilities",
    "bigram models are simple but effective",
]
print(BigramModel(test_corpus).generate_text("bigram", 5))

bigram probabilities


In [ ]:
# 7. Create a Dockerfile

In [ ]:
%%writefile Dockerfile
FROM python:3.12-slim

WORKDIR /app

COPY --from=ghcr.io/astral-sh/uv:latest /uv /uvx /bin/

COPY pyproject.toml uv.lock ./
RUN uv sync --frozen --no-dev

COPY app ./app
COPY main.py .

EXPOSE 80

CMD ["uv", "run", "--no-dev", "fastapi", "run", "main.py", "--host", "0.0.0.0", "--port", "80"]

Overwriting Dockerfile


In [ ]:
%%writefile .dockerignore
.venv
__pycache__/
*.pyc
.git
.gitignore
.ipynb_checkpoints
*.ipynb
.DS_Store


Overwriting .dockerignore


In [ ]:
# Run in a terminal (blocks the notebook if run here):
# docker run -p 8000:80 sps-genai

Once the container is running (terminal), test both endpoints from here:

In [ ]:
import requests

base = "http://127.0.0.1:8000"   # works for `docker run -p 8000:80 ...` and `uv run fastapi dev main.py`
try:
    print(requests.post(f"{base}/generate", json={"start_word": "bigram", "length": 5}, timeout=10).json())
    r = requests.post(f"{base}/embedding", json={"word": "king"}, timeout=10).json()
    print(r["word"], r["dimensions"], r["embedding"][:3])
except requests.exceptions.ConnectionError:
    print("Nothing is listening on port 8000. Start the app in a Terminal first "
          "(`uv run fastapi dev main.py`, or `docker run -p 8000:80 sps-genai`), then re-run this cell.")

## 9. Version control with git
**What went wrong before, and the fix**
- `%cd ..` moved *out* of `sps_genai`, so `git init` / `git add .` ran in `1 API` and tried to add every notebook, PDF and folder in it.
- `fatal: mmap failed: Operation timed out` and the hang on `git remote add` come from keeping a git repo inside **OneDrive**: cloud-only files and the syncing `.git` folder make git's file reads time out (and can corrupt the repo).
- `git push` over HTTPS asks for credentials; a notebook cell can't answer that prompt, so it hangs.

So the repo lives **outside OneDrive** (`~/projects/sps_genai`), contains only `sps_genai`, and the push fails fast instead of hanging. `uv.lock` **is** committed (Docker needs it for `uv sync --frozen`); `.venv` is not.

In [ ]:
import os

# Code stays in OneDrive; the git repo is a clean copy outside it.
src_dir  = project_dir if "project_dir" in globals() else os.getcwd()
repo_dir = os.path.expanduser("~/projects/sps_genai")

assert os.path.isfile(os.path.join(src_dir, "main.py")) and os.path.isdir(os.path.join(src_dir, "app")), \
    f"{src_dir} is not the sps_genai project root (needs main.py and app/)."

os.makedirs(repo_dir, exist_ok=True)
# Copy code only: no .venv (`uv sync` recreates it), no old .git, no caches
import shutil
shutil.copytree(src_dir, repo_dir, dirs_exist_ok=True,
                ignore=shutil.ignore_patterns(".venv", ".git", "__pycache__", ".ipynb_checkpoints"))
%cd "{repo_dir}"
!ls -A

In [ ]:
%%writefile .gitignore
.venv/
__pycache__/
*.pyc
.ipynb_checkpoints/
.DS_Store
.env


In [ ]:
import subprocess

def sh(cmd):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True).stdout.strip()

# git needs an identity to commit
if not sh("git config user.name") or not sh("git config user.email"):
    print("Set once in Terminal, then re-run:\n"
          "  git config --global user.name  'Your Name'\n"
          "  git config --global user.email 'you@example.com'")
else:
    print("git identity:", sh("git config user.name"), "<" + sh("git config user.email") + ">")

In [ ]:
# Safety: never init/add anywhere except the sps_genai repo copy
assert os.path.samefile(os.getcwd(), repo_dir), f"Expected to be in {repo_dir}, but in {os.getcwd()}"

if not os.path.isdir(".git"):
    !git init -b main
!git add .
!git status --short

In [ ]:
!git commit -m "FastAPI bigram text generation and spaCy embedding API with Docker"

In [ ]:
REMOTE = "https://github.com/ai2584987/appliedgenai.git"

# Replace origin if it already exists, so re-running never errors
!git remote remove origin 2>/dev/null; git remote add origin {REMOTE}
!git remote -v

**Push.** GitHub no longer accepts account passwords over HTTPS: you need a personal access token or `gh auth login`. `GIT_TERMINAL_PROMPT=0` makes the cell fail immediately with an auth error instead of hanging. If it fails, run the push once in **Terminal** (`cd ~/projects/sps_genai && git push -u origin main`), enter the token when asked, and macOS Keychain will remember it. Also make sure the GitHub repo is empty (no README) or use `git pull --rebase origin main` first.

In [ ]:
!GIT_TERMINAL_PROMPT=0 git push -u origin main

---
## Prompting demos (FLAN-T5)
Your original prompting-technique cells, unchanged apart from this section header.

In [ ]:
# Install dependencies (run once). 
!pip install -q -U transformers torch sentencepiece

import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/flan-t5-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
model.eval()

def ask(prompt, max_new_tokens=60):
    # Greedy decoding by default -> clean, reproducible answers for demos.
    # (T5 returns only the answer, not prompt + continuation, so no slicing needed.)
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        output_ids = model.generate(**inputs, max_new_tokens=max_new_tokens)
    print(tokenizer.decode(output_ids[0], skip_special_tokens=True).strip())


In [ ]:
# ===============================
# 1. Zero-shot Prompting
# ===============================
print("---- ZERO SHOT ----")
prompt = """ identify if a number is even or odd.
Question: number is: 1
Answer: {"label": "odd"}
Question: number is: 2
Answer: {"label": "even"}
"""
ask("Classify the number as odd or even. Number: 9'")

In [ ]:
# ===============================
# 2. One-shot Prompting
# ===============================
print("---- ONE SHOT ----")
prompt = """Classify the topic of these sentences (topics: sports, politics, tech).

Sentence: The game was terrible.
Sentiment: Sports

Sentence: I really disliked the company.
Sentiment:"""
ask(prompt)

In [ ]:
# ===============================
# 3. Few-shot Prompting
# ===============================
prompt = """Extract the names from these sentences.

Sentence: Mathew loves pizza.
Sentiment: Mathew

Sentence: While building a bike, Abi injured her toe.
Sentiment: Abi

Sentence: Between the auditorium and the dance floor Kelly got lost.
Sentiment:"""
ask(prompt)

In [ ]:
# ===============================
# 4. Chain-of-Thought Prompting
# (Model is guided to reason step by step)
# ===============================
print("---- CHAIN OF THOUGHT ----")
prompt = """Question: If a train travels 60km in 1 hour, how far can it travel in 3 hours?
Let's think step by step."""
ask(prompt)

In [ ]:
# ===============================
# 5. Role Prompting (persona: junior school tutor)
# ===============================
print("---- ROLE PROMPT: junior school tutor ----")
prompt = """You are the school teacher for a classroom of 7 year old students.
Explain the moon to these 7 year old students in simple words."""
ask(prompt, max_new_tokens=80)